In [78]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import re
import string

import pandas as pd

import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [79]:
# =========================================================
# DOWNLOAD NLTK RESOURCES
# =========================================================

nltk.download("stopwords")

nltk.download("punkt")

nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [80]:
# =========================================================
# LOAD DATASET
# =========================================================

DATA_PATH = (

    "../data/processed/"
    "emails_dataset.csv"
)

df = pd.read_csv(

    DATA_PATH,
    keep_default_na=False
)

print(

    df.shape
)

df.head()

(5767, 8)


,label,subject,from,to,date,content_type,body,email_length
0,ham,Re: New Sequences Window,Robert Elz <kre@munnari.OZ.AU>,Chris Garrigues <cwg-dated-1030377287.06fa6d@D...,"Thu, 22 Aug 2002 18:26:25 +0700","text/plain; charset=""us-ascii""","Date: Wed, 21 Aug 2002 10:54:46 -0500\n...",1598
1,ham,[zzzzteana] RE: Alexander,Steve Burt <Steve_Burt@cursor-system.com>,"""'zzzzteana@yahoogroups.com'"" <zzzzteana@yahoo...","Thu, 22 Aug 2002 12:46:18 +0100","text/plain; charset=""US-ASCII""","Martin A posted:\nTassos Papadopoulos, the Gre...",894
2,ham,[zzzzteana] Moscow bomber,Tim Chapman <timc@2ubh.com>,zzzzteana <zzzzteana@yahoogroups.com>,"Thu, 22 Aug 2002 13:52:38 +0100","text/plain; charset=""US-ASCII""",Man Threatens Explosion In Moscow \n\nThursday...,1746
3,ham,[IRR] Klez: The Virus That Won't Die,Monty Solomon <monty@roscom.com>,undisclosed-recipient:;,"Thu, 22 Aug 2002 09:15:25 -0400","text/plain; charset=""us-ascii""",Klez: The Virus That Won't Die\n \nAlready the...,1125
4,ham,Re: [zzzzteana] Nothing like mama used to make,Stewart Smith <Stewart.Smith@ee.ed.ac.uk>,zzzzteana@yahoogroups.com,"Thu, 22 Aug 2002 14:38:22 +0100","text/plain; charset=""US-ASCII""","> in adding cream to spaghetti carbonara, whi...",1047


In [81]:
df.isnull().sum()

label           0
subject         0
from            0
to              0
date            0
content_type    0
body            0
email_length    0
dtype: int64

In [82]:
# =========================================================
# NLP OBJECTS
# =========================================================

stop_words = set(

    stopwords.words("english")
)

stemmer = PorterStemmer()

In [83]:
# =========================================================
# BASIC TEXT CLEANING
# =========================================================

def clean_text(text):

    """
    Cleans raw email text.

    Steps:
    - validate input
    - lowercase conversion
    - URL removal
    - email address removal
    - HTML removal
    - HTML noise cleanup
    - digit removal
    - punctuation removal
    - whitespace normalization

    Args:
        text (str):
            Raw email text.

    Returns:
        str:
            Cleaned text.
    """

    # =============================================
    # HANDLE NON-STRING INPUTS
    # =============================================

    if not isinstance(text, str):

        return ""

    # =============================================
    # LOWERCASE
    # =============================================

    text = text.lower()

    # =============================================
    # REMOVE URLS
    # =============================================

    text = re.sub(

        r"http\S+|www\S+",

        " ",

        text
    )

    # =============================================
    # REMOVE EMAIL ADDRESSES
    # =============================================

    text = re.sub(

        r"\S+@\S+",

        " ",

        text
    )

    # =============================================
    # REMOVE HTML TAGS
    # =============================================

    text = re.sub(

        r"<.*?>",

        " ",

        text
    )

    # =============================================
    # REMOVE COMMON HTML ARTIFACT WORDS
    # =============================================

    html_noise = [

        "nbsp",
        "href",
        "font",
        "color",
        "html",
        "http"
    ]

    for noise in html_noise:

        text = text.replace(

            noise,

            " "
        )

    # =============================================
    # REMOVE DIGITS
    # =============================================

    text = re.sub(

        r"\d+",

        " ",

        text
    )

    # =============================================
    # REMOVE PUNCTUATION
    # =============================================

    text = text.translate(

        str.maketrans(

            "",

            "",

            string.punctuation
        )
    )

    # =============================================
    # REMOVE EXTRA WHITESPACES
    # =============================================

    text = re.sub(

        r"\s+",

        " ",

        text
    )

    return text.strip()

In [84]:
# =========================================================
# TOKENIZATION
# =========================================================

def tokenize_text(text):

    """
    Converts text into word tokens.

    Args:
        text (str):
            Cleaned text.

    Returns:
        list:
            List of word tokens.
    """

    return word_tokenize(text)

In [85]:

# =========================================================
# CUSTOM STOPWORDS
# =========================================================

"""
Adds dataset-specific noisy words
that are not useful for prediction.

Examples:
- mailing-list artifacts
- generic low-information words
- repeated dataset-specific tokens
"""

custom_stopwords = {

    "mv",
    "im",
    "us",
    "one",
    "would",
    "could",
    "also",
    "get",
    "like"
}

stop_words.update(

    custom_stopwords
)

print(

    f"Total Stopwords: {len(stop_words)}"
)


# =========================================================
# STOPWORD REMOVAL
# =========================================================

def remove_stopwords(tokens):

    """
    Removes:
    - common English stopwords
    - custom noisy words
    - single-character noise tokens

    Args:
        tokens (list):
            List of word tokens.

    Returns:
        list:
            Filtered tokens.
    """

    filtered_tokens = [

        word

        for word in tokens

        if word not in stop_words
        and len(word) > 1
    ]

    return filtered_tokens

Total Stopwords: 207


In [86]:
# =========================================================
# STEMMING
# =========================================================

def stem_tokens(tokens):

    """
    Applies stemming to tokens.

    Args:
        tokens (list):
            Word tokens.

    Returns:
        list:
            Stemmed tokens.
    """

    stemmed_tokens = [

        stemmer.stem(word)

        for word in tokens
    ]

    return stemmed_tokens

In [87]:
# =========================================================
# COMPLETE NLP PIPELINE
# =========================================================

def preprocess_text(text):

    """
    Complete NLP preprocessing pipeline.

    Steps:
    - clean text
    - tokenize
    - remove stopwords
    - apply stemming

    Args:
        text (str):
            Raw email text.

    Returns:
        str:
            Fully processed text.
    """

    # =============================================
    # CLEAN TEXT
    # =============================================

    text = clean_text(text)

    # =============================================
    # TOKENIZE
    # =============================================

    tokens = tokenize_text(text)

    # =============================================
    # REMOVE STOPWORDS
    # =============================================

    tokens = remove_stopwords(tokens)

    # =============================================
    # STEM TOKENS
    # =============================================

    tokens = stem_tokens(tokens)

    # =============================================
    # REJOIN TOKENS
    # =============================================

    processed_text = " ".join(tokens)

    return processed_text

In [88]:
# =========================================================
# TEST NLP PIPELINE
# =========================================================

sample_email = df["body"].iloc[0]

processed_email = preprocess_text(

    sample_email
)

print(

    processed_email[:2000]
)

date wed aug chri garrigu messageid cant reproduc error repeat everi time without fail debug log pick happen pickit exec pick inbox list lbrace lbrace subject ftp rbrace rbrace sequenc mercuri exec pick inbox list lbrace lbrace subject ftp rbrace rbrace sequenc mercuri ftocpickmsg hit mark hit tkerror syntax error express int note run pick command hand delta pick inbox list lbrace lbrace subject ftp rbrace rbrace sequenc mercuri hit that hit come obvious version nmh use delta pick version pick nmh compil fuchsiacsmuozau sun mar ict relev part mhprofil delta mhparam pick seq sel list sinc pick command work sequenc actual that explicit command line search popup come mhprofil creat kre ps still use version code form day ago havent abl reach cv repositori today local rout issu think exmhwork mail list


In [89]:
# =========================================================
# APPLY NLP PIPELINE
# =========================================================

df["clean_text"] = (

    df["body"]

    .apply(preprocess_text)
)

In [90]:
# =========================================================
# COMPARE RAW VS CLEANED TEXT
# =========================================================

comparison_df = pd.DataFrame({

    "raw_text":

        df["body"].head(3),

    "clean_text":

        df["clean_text"].head(3)
})

comparison_df

,raw_text,clean_text
0,"Date: Wed, 21 Aug 2002 10:54:46 -0500\n...",date wed aug chri garrigu messageid cant repro...
1,"Martin A posted:\nTassos Papadopoulos, the Gre...",martin post tasso papadopoulo greek sculptor b...
2,Man Threatens Explosion In Moscow \n\nThursday...,man threaten explos moscow thursday august pm ...


In [91]:
# =========================================================
# CHECK EMPTY CLEANED EMAILS
# =========================================================

empty_cleaned = df[

    df["clean_text"]

    .str.strip()

    == ""
]

print(

    f"Empty cleaned emails: {len(empty_cleaned)}"
)

Empty cleaned emails: 7


In [92]:
# =========================================================
# REMOVE EMPTY CLEANED EMAILS
# =========================================================

df = df[

    df["clean_text"]

    .str.strip()

    != ""
]

print(

    df.shape
)

(5760, 9)


In [93]:
# =========================================================
# CLEANED TEXT LENGTH FEATURE
# =========================================================

df["clean_text_length"] = (

    df["clean_text"]

    .apply(len)
)

df["clean_text_length"].describe()

count      5760.000000
mean        904.227951
std        2677.552697
min           4.000000
25%         252.000000
50%         464.000000
75%         813.000000
max      109915.000000
Name: clean_text_length, dtype: float64

In [94]:
# =========================================================
# REMOVE EMPTY CLEANED EMAILS
# =========================================================

initial_rows = len(df)

df = df[

    df["clean_text"]

    .str.strip()

    != ""
]

final_rows = len(df)

print(

    f"Removed {initial_rows - final_rows} empty cleaned emails."
)

Removed 0 empty cleaned emails.


In [95]:
# =========================================================
# REMOVE VERY SHORT CLEANED EMAILS
# =========================================================

initial_rows = len(df)

df = df[

    df["clean_text_length"] >= 10
]

final_rows = len(df)

print(

    f"Removed {initial_rows - final_rows} very short emails."
)

Removed 5 very short emails.


In [96]:
# =========================================================
# EXPORT CLEANED DATASET
# =========================================================

output_path = (

    "../data/interim/"
    "emails_cleaned.csv"
)

df.to_csv(

    output_path,

    index=False
)

print(

    f"Cleaned dataset saved to:\n{output_path}"
)

Cleaned dataset saved to:
../data/interim/emails_cleaned.csv


In [97]:
# =========================================================
# VERIFY CLEANED DATASET EXPORT
# =========================================================

saved_df = pd.read_csv(

    output_path,

    keep_default_na=False
)

print(

    saved_df.shape
)

saved_df.head()

(5755, 10)


,label,subject,from,to,date,content_type,body,email_length,clean_text,clean_text_length
0,ham,Re: New Sequences Window,Robert Elz <kre@munnari.OZ.AU>,Chris Garrigues <cwg-dated-1030377287.06fa6d@D...,"Thu, 22 Aug 2002 18:26:25 +0700","text/plain; charset=""us-ascii""","Date: Wed, 21 Aug 2002 10:54:46 -0500\n...",1598,date wed aug chri garrigu messageid cant repro...,808
1,ham,[zzzzteana] RE: Alexander,Steve Burt <Steve_Burt@cursor-system.com>,"""'zzzzteana@yahoogroups.com'"" <zzzzteana@yahoo...","Thu, 22 Aug 2002 12:46:18 +0100","text/plain; charset=""US-ASCII""","Martin A posted:\nTassos Papadopoulos, the Gre...",894,martin post tasso papadopoulo greek sculptor b...,399
2,ham,[zzzzteana] Moscow bomber,Tim Chapman <timc@2ubh.com>,zzzzteana <zzzzteana@yahoogroups.com>,"Thu, 22 Aug 2002 13:52:38 +0100","text/plain; charset=""US-ASCII""",Man Threatens Explosion In Moscow \n\nThursday...,1746,man threaten explos moscow thursday august pm ...,902
3,ham,[IRR] Klez: The Virus That Won't Die,Monty Solomon <monty@roscom.com>,undisclosed-recipient:;,"Thu, 22 Aug 2002 09:15:25 -0400","text/plain; charset=""us-ascii""",Klez: The Virus That Won't Die\n \nAlready the...,1125,klez viru wont die alreadi prolif viru ever kl...,590
4,ham,Re: [zzzzteana] Nothing like mama used to make,Stewart Smith <Stewart.Smith@ee.ed.ac.uk>,zzzzteana@yahoogroups.com,"Thu, 22 Aug 2002 14:38:22 +0100","text/plain; charset=""US-ASCII""","> in adding cream to spaghetti carbonara, whi...",1047,ad cream spaghetti carbonara effect pasta make...,464


In [98]:
# =========================================================
# CHECK MISSING VALUES
# =========================================================

saved_df.isnull().sum()

label                0
subject              0
from                 0
to                   0
date                 0
content_type         0
body                 0
email_length         0
clean_text           0
clean_text_length    0
dtype: int64

In [99]:
df["clean_text_length"].describe()

count      5755.000000
mean        905.008341
std        2678.584808
min          12.000000
25%         252.000000
50%         464.000000
75%         813.000000
max      109915.000000
Name: clean_text_length, dtype: float64